# Notebook 2: Predicting Price and Quantities

**Self-contained.** This notebook loads the dataset directly from HuggingFace.

We evaluate how well different feature sets and models predict:
- **Quantity** (`Q_t` = sales rank)
- **Price** (`P_bb_t` = buy-box price)

Good out-of-sample R² on both is a prerequisite for the DML causal estimator in Notebook 3.

**Models:** OLS and LightGBM across three feature specifications:
1. Lagged variables + tabular controls
2. + PCA of text embeddings (5 components)
3. + Cluster similarity scores (5 clusters)

In [ ]:
import re
import datasets
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

from lightgbm import LGBMRegressor
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics import pairwise_distances, r2_score

palette = sns.color_palette("colorblind")

## 1. Load and Prepare Data

In [ ]:
ds = datasets.load_dataset("janteichertkluge/demand-analysis-repro")
df_train = ds["train"].to_pandas()
df_test  = ds["test"].to_pandas()

# Canonical names
df_train = df_train.rename(columns={"SALES_RANK": "Q_t", "BUYBOX_PRICE": "P_bb_t"})
df_test  = df_test.rename(columns={"SALES_RANK": "Q_t", "BUYBOX_PRICE": "P_bb_t"})

df_train["date"] = df_train["date"].astype(str)
df_test["date"]  = df_test["date"].astype(str)

print(f"Train: {df_train.shape}  |  Test: {df_test.shape}")

In [ ]:
# Embedding columns
emb_cols = sorted([c for c in df_train.columns if c.startswith("emb_")])
print(f"Embedding dimensions: {len(emb_cols)}")

### 1a. Process Embeddings → PCA and Cluster Similarities

In [ ]:
# Center and L2-normalize using train statistics
emb_tr = df_train[emb_cols].values.astype(float)
emb_te = df_test[emb_cols].values.astype(float)

global_mean = emb_tr.mean(axis=0)
emb_tr_n = normalize(emb_tr - global_mean, axis=1)
emb_te_n = normalize(emb_te - global_mean, axis=1)

# PCA (fit on train)
pca = PCA(n_components=5, random_state=42)
pca.fit(emb_tr_n)
pca_tr = pca.transform(emb_tr_n)
pca_te = pca.transform(emb_te_n)
pca_cols = [f"pca_{i}" for i in range(5)]
df_train[pca_cols] = pca_tr
df_test[pca_cols]  = pca_te

# K-Means clusters (fit on train)
km = KMeans(n_clusters=5, random_state=42, n_init=50)
km.fit(emb_tr_n)
centroids = km.cluster_centers_

sim_cols = [f"sim_cluster_{i}" for i in range(5)]
for j, c in enumerate(centroids):
    df_train[sim_cols[j]] = 1.0 - pairwise_distances(emb_tr_n, c.reshape(1,-1), metric="cosine").ravel()
    df_test[sim_cols[j]]  = 1.0 - pairwise_distances(emb_te_n, c.reshape(1,-1), metric="cosine").ravel()

print("PCA and similarity features added.")

### 1b. Lagged Variables and Dummies

In [ ]:
def add_lags(df, vars_, n=1):
    df = df.sort_values(["ASIN", "date"]).copy()
    for v in vars_:
        for i in range(1, n+1):
            df[f"{v}_lag{i}"] = df.groupby("ASIN")[v].shift(i)
    return df

df_train = add_lags(df_train, ["Q_t", "P_bb_t", "REVIEW_COUNT", "RATING"])
df_test  = add_lags(df_test,  ["Q_t", "P_bb_t", "REVIEW_COUNT", "RATING"])

# Subcategory dummies
if "subcat_aggregated" in df_train.columns:
    sub_tr = pd.get_dummies(df_train["subcat_aggregated"], prefix="sub", drop_first=True).astype(int)
    sub_te = pd.get_dummies(df_test["subcat_aggregated"],  prefix="sub", drop_first=True).astype(int)
    sub_te = sub_te.reindex(columns=sub_tr.columns, fill_value=0)
    df_train[sub_tr.columns] = sub_tr.values
    df_test[sub_te.columns]  = sub_te.values
    subcat_cols = sub_tr.columns.tolist()
else:
    subcat_cols = []

# Time dummies
time_tr = pd.get_dummies(df_train["date"], prefix="t", drop_first=True).astype(int)
time_te = pd.get_dummies(df_test["date"],  prefix="t", drop_first=True).astype(int)
time_te = time_te.reindex(columns=time_tr.columns, fill_value=0)
df_train[time_tr.columns] = time_tr.values
df_test[time_te.columns]  = time_te.values
time_cols = time_tr.columns.tolist()

# Drop rows with missing lags
df_train = df_train.dropna(subset=["Q_t_lag1", "P_bb_t_lag1"]).reset_index(drop=True)
df_test  = df_test.dropna(subset=["Q_t_lag1", "P_bb_t_lag1"]).reset_index(drop=True)

print(f"After lags  →  Train: {df_train.shape}  |  Test: {df_test.shape}")

## 2. Define Feature Specifications

In [ ]:
outcome   = "Q_t"
treatment = "P_bb_t"

cont_controls_candidates = [
    "RATING_lag1", "REVIEW_COUNT_lag1",
    "New Offer Count: Current",
    "Count of retrieved live offers: New, FBA",
    "Count of retrieved live offers: New, FBM",
    "Lightning Deals: Upcoming Deal",
    "Buy Box: Is FBA",
]
cont_controls = [c for c in cont_controls_candidates if c in df_train.columns]
lag_controls  = ["Q_t_lag1", "P_bb_t_lag1"]
base_controls = lag_controls + cont_controls + subcat_cols + time_cols

feature_specs = {
    "Tabular":       base_controls,
    "Tabular + PCA": base_controls + pca_cols,
    "Tabular + Sim": base_controls + sim_cols,
}
print(f"Base controls: {len(base_controls)}  |  PCA: {len(pca_cols)}  |  Sim: {len(sim_cols)}")

## 3. Helper Functions

In [ ]:
def get_X_ols(df, controls):
    cols = [c for c in controls if c in df.columns]
    X = df[cols].fillna(0).copy()
    X.columns = [re.sub(r"[^A-Za-z0-9_]+", "", c) for c in X.columns]
    return sm.add_constant(X, has_constant="add")

def get_X_lgbm(df, controls):
    cols = [c for c in controls if c in df.columns]
    X = df[cols].fillna(0).copy()
    X.columns = [re.sub(r"[^A-Za-z0-9_]+", "", c) for c in X.columns]
    return X

def r2(y, yhat):
    return r2_score(y, yhat)

column_names = ["R² Q Train", "R² Q Test", "R² P Train", "R² P Test"]

## 4. Fit Models and Evaluate

In [ ]:
results = []

for spec_name, controls in feature_specs.items():
    print(f"\n{'='*60}")
    print(f"  Specification: {spec_name}")
    print(f"{'='*60}")

    # --- OLS ---
    X_tr = get_X_ols(df_train, controls)
    X_te = get_X_ols(df_test,  controls)
    X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)

    ols_q = sm.OLS(df_train[outcome],   X_tr).fit()
    ols_p = sm.OLS(df_train[treatment], X_tr).fit()

    r2_q_tr = r2(df_train[outcome],   ols_q.predict(X_tr))
    r2_q_te = r2(df_test[outcome],    ols_q.predict(X_te))
    r2_p_tr = r2(df_train[treatment], ols_p.predict(X_tr))
    r2_p_te = r2(df_test[treatment],  ols_p.predict(X_te))
    print(f"  OLS      Q:  Train {r2_q_tr:.4f}  Test {r2_q_te:.4f}    "
          f"P:  Train {r2_p_tr:.4f}  Test {r2_p_te:.4f}")
    results.append({"Spec": spec_name, "Model": "OLS",
                    "R² Q Train": r2_q_tr, "R² Q Test": r2_q_te,
                    "R² P Train": r2_p_tr, "R² P Test": r2_p_te})

    # --- LightGBM ---
    Xl_tr = get_X_lgbm(df_train, controls)
    Xl_te = get_X_lgbm(df_test,  controls)
    Xl_te = Xl_te.reindex(columns=Xl_tr.columns, fill_value=0)

    lgbm_q = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    lgbm_q.fit(Xl_tr, df_train[outcome])
    lgbm_p = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    lgbm_p.fit(Xl_tr, df_train[treatment])

    r2_q_tr = r2(df_train[outcome],   lgbm_q.predict(Xl_tr))
    r2_q_te = r2(df_test[outcome],    lgbm_q.predict(Xl_te))
    r2_p_tr = r2(df_train[treatment], lgbm_p.predict(Xl_tr))
    r2_p_te = r2(df_test[treatment],  lgbm_p.predict(Xl_te))
    print(f"  LightGBM Q:  Train {r2_q_tr:.4f}  Test {r2_q_te:.4f}    "
          f"P:  Train {r2_p_tr:.4f}  Test {r2_p_te:.4f}")
    results.append({"Spec": spec_name, "Model": "LightGBM",
                    "R² Q Train": r2_q_tr, "R² Q Test": r2_q_te,
                    "R² P Train": r2_p_tr, "R² P Test": r2_p_te})

df_results = pd.DataFrame(results).set_index(["Spec", "Model"])

## 5. Summary Table (Test R² in %)

In [ ]:
(df_results[["R² Q Test", "R² P Test"]] * 100).round(2)

## 6. Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

df_plot = df_results.reset_index()
df_plot["label"] = df_plot["Spec"] + "\n(" + df_plot["Model"] + ")"
x = np.arange(len(df_plot))
w = 0.35

ax.bar(x - w/2, df_plot["R² Q Test"] * 100, w, label="Quantity (Q_t)",  color=palette[0])
ax.bar(x + w/2, df_plot["R² P Test"] * 100, w, label="Price (P_bb_t)", color=palette[1])

ax.set_xticks(x)
ax.set_xticklabels(df_plot["label"], fontsize=9)
ax.set_ylabel("Out-of-sample R² (%)")
ax.set_title("Predictive Performance: Test-Set R²")
ax.legend()
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.show()